# D03 — Holdout specifications

Generates the deterministic 30-perturbation holdout specifications used by P01 (CPA) and P02 (GEARS). Two split strategies are produced per cell type:
- **random** — uniform random draw of 30 perturbations from the 150-perturbation pool, per seed.
- **stratified** — leverage-quintile stratified draw (6 perturbations per quintile), per seed.

**Manuscript reference:** Methods §2.3 (Holdout strategies).

For the primary CPA sweep, seeds 1 through 100 are generated under each strategy (200 30-perturbation holdouts per cell type). Seeds 1 through 7 are reused for the matched-seed GEARS audit in P02.

**Inputs:**
- `data/severity_refs/replogle_K562_severity.csv` (from D02)
- `data/severity_refs/replogle_RPE1_severity.csv` (from D02)

**Outputs:**
- `data/holdout_specs/holdout_specs.csv` — one row per (cell_type, split_type, seed, position), with the `perturbation_target` for each of the 30 holdouts.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
from perturb_style import DATA_DIR

SEVERITY_REFS = {
    "K562": DATA_DIR / "severity_refs" / "replogle_K562_severity.csv",
    "RPE1": DATA_DIR / "severity_refs" / "replogle_RPE1_severity.csv",
}
OUT_PATH = DATA_DIR / "holdout_specs" / "holdout_specs.csv"
OUT_PATH.parent.mkdir(exist_ok=True, parents=True)

POOL_SIZE = 150     # perturbations randomly drawn from each atlas to form the pool
HOLDOUT_SIZE = 30   # per holdout
N_SEEDS = 100       # primary CPA sweep


## Check that severity references exist

In [ ]:
missing = [c for c, p in SEVERITY_REFS.items() if not p.exists()]
if missing:
    print(f"Missing severity reference(s): {missing}")
    print(f"Expected location: {DATA_DIR / 'severity_refs'}/")
    print()
    print("Run D02 first to generate them from the Replogle supplementary tables,")
    print("or use ./run_all.sh --figures for the precomputed-CSV reproducibility path.")
else:
    print(f"Both severity references found.")
    for cell, path in SEVERITY_REFS.items():
        n = len(pd.read_csv(path))
        print(f"  {cell}: {path.name}  ({n} gene-level rows)")


## Generate holdout specifications

For each cell type, draw a fixed 150-perturbation pool with seed 0, then for each of seeds 1..100 draw either a random 30-perturbation subset or a leverage-quintile-stratified 30-perturbation subset. The holdout assignments are deterministic and reproducible.

In [ ]:
if not missing:
    rows = []

    for cell_type, sev_path in SEVERITY_REFS.items():
        sev = pd.read_csv(sev_path).dropna(subset=["leverage_score"]).reset_index(drop=True)

        # Fixed pool: draw POOL_SIZE perturbations with seed 0 (per Methods §2.2)
        rng_pool = np.random.default_rng(0)
        pool_idx = rng_pool.choice(len(sev), size=POOL_SIZE, replace=False)
        pool = sev.iloc[pool_idx].sort_values("leverage_score").reset_index(drop=True)
        print(f"{cell_type}: built {POOL_SIZE}-perturbation pool")
        print(f"  leverage range in pool: {pool.leverage_score.min():.3f} to {pool.leverage_score.max():.3f}")

        # Quintile assignment (5 bins, 30 perturbations each) for stratified draws
        pool["quintile"] = pd.qcut(pool["leverage_score"], q=5, labels=False)

        for split_type in ("random", "stratified"):
            for seed in range(1, N_SEEDS + 1):
                rng = np.random.default_rng(seed)
                if split_type == "random":
                    pick = pool.sample(HOLDOUT_SIZE, random_state=seed).reset_index(drop=True)
                else:  # stratified: 6 from each of 5 quintiles
                    parts = []
                    for q in range(5):
                        bin_df = pool[pool["quintile"] == q]
                        # Use a quintile-specific sub-seed so each quintile draw is independent
                        sub = bin_df.sample(HOLDOUT_SIZE // 5, random_state=seed * 10 + q)
                        parts.append(sub)
                    pick = pd.concat(parts).reset_index(drop=True)

                for pos, row in enumerate(pick.itertuples(index=False), start=1):
                    rows.append({
                        "cell_type": cell_type,
                        "split_type": split_type,
                        "seed": seed,
                        "position": pos,
                        "perturbation_target": row.perturbation_target,
                        "leverage_score": float(row.leverage_score),
                    })

    spec = pd.DataFrame(rows)
    spec.to_csv(OUT_PATH, index=False)

    print(f"\nWrote {OUT_PATH} ({len(spec)} rows)")
    print(f"Expected: 2 cell types x 2 splits x {N_SEEDS} seeds x {HOLDOUT_SIZE} perts = {2 * 2 * N_SEEDS * HOLDOUT_SIZE}")
    print()
    print("Per-condition row counts:")
    print(spec.groupby(["cell_type", "split_type"]).size().to_string())
